In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from datetime import datetime, timedelta
import ipywidgets as widgets
from IPython.display import display

# -------------------------------
# 1. GENERATE SYNTHETIC DATA
# -------------------------------

np.random.seed(42)

days = 60
hours = days * 24

dates = [datetime(2025,1,1) + timedelta(hours=i) for i in range(hours)]

day_types = []
energy = []

for d in range(days):
   
    if d % 15 == 0:
        day_type = "Event"
        base = 450
    elif d % 7 == 0:
        day_type = "Exam"
        base = 350
    else:
        day_type = "Normal"
        base = 250
       
    for h in range(24):
       
        hourly_pattern = 50*np.sin((h/24)*2*np.pi) + 100
       
        value = base + hourly_pattern + np.random.normal(0,20)
       
        energy.append(value)
        day_types.append(day_type)

df = pd.DataFrame({
    "timestamp":dates,
    "energy":energy,
    "day_type":day_types
})

df["hour"]=df["timestamp"].dt.hour

# -------------------------------
# 2. PREPARE DATA FOR LSTM
# -------------------------------

scaler = MinMaxScaler()

scaled_energy = scaler.fit_transform(df[["energy"]])

sequence_length = 24

X=[]
y=[]

for i in range(len(scaled_energy)-sequence_length):
   
    X.append(scaled_energy[i:i+sequence_length])
    y.append(scaled_energy[i+sequence_length])

X = np.array(X)
y = np.array(y)

split = int(len(X)*0.8)

X_train = X[:split]
X_test = X[split:]

y_train = y[:split]
y_test = y[split:]

# -------------------------------
# 3. BUILD SIMPLE LSTM MODEL
# -------------------------------

model = Sequential()

model.add(LSTM(32,input_shape=(24,1)))
model.add(Dense(1))

model.compile(optimizer="adam",loss="mse")

model.fit(X_train,y_train,epochs=5,batch_size=32,verbose=0)

# -------------------------------
# 4. PREDICTIONS
# -------------------------------

predictions = model.predict(X_test)

predictions = scaler.inverse_transform(predictions)

actual = scaler.inverse_transform(y_test)

result_df = df.iloc[sequence_length+split:].copy()

result_df["actual_energy"]=actual
result_df["predicted_energy"]=predictions

# -------------------------------
# 5. DASHBOARD FUNCTION
# -------------------------------

def plot_daytype(daytype):
   
    filtered = result_df[result_df["day_type"]==daytype]
   
    fig = go.Figure()
   
    fig.add_trace(go.Scatter(
        x=filtered["timestamp"],
        y=filtered["actual_energy"],
        mode="lines",
        name="Actual Energy"
    ))
   
    fig.add_trace(go.Scatter(
        x=filtered["timestamp"],
        y=filtered["predicted_energy"],
        mode="lines",
        name="Predicted Energy"
    ))
   
    fig.update_layout(
        title=f"Electricity Forecast (Day Type: {daytype})",
        xaxis_title="Time",
        yaxis_title="Energy Consumption",
        height=500
    )
   
    fig.show()

# -------------------------------
# 6. INTERACTIVE FILTER
# -------------------------------

dropdown = widgets.Dropdown(
    options=result_df["day_type"].unique(),
    description="Day Type:"
)

widgets.interact(plot_daytype,daytype=dropdown)

# -------------------------------
# SAMPLE OUTPUT PRINT
# -------------------------------

print("\nSample Forecast Data:\n")
print(result_df[["timestamp","day_type","actual_energy","predicted_energy"]].head())

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

ModuleNotFoundError: No module named 'ml_dtypes'